# Breakthrough curve analysis

When you load protein onto a packed column, the resin near the inlet saturates first and a concentration front moves down the column. Protein appears in the outlet when the front reaches the bottom: this is **breakthrough**. The volume at which it happens tells you how much protein the column holds, and the steepness of the curve tells you how fast protein moves from the liquid into the resin.

This notebook works through seven parts:

1. **Plot** your breakthrough curves.
2. **Calculate the column capacity** from a mass balance on each run.
3. **Fit the mass-transfer coefficient** $K_a$ by comparing a simulation of the column with your data.
4. **Report a 95% confidence interval** for $K_a$.
5. **Check the interval** by plotting how the fit worsens as $K_a$ moves away from its best value.
6. **Check the residuals** for patterns the model does not capture.
7. **Summarize** the results in a table.

**To use it:** put your data file in the same folder as this notebook and check the settings cell. Then choose **Run → Run All Cells** from the menu. After changing a setting, run all cells again.

## Settings

The data file needs one row per fraction, with the cumulative volume collected, the protein concentration in that fraction, the flow rate, the inlet (feed) concentration and a run name. Use your **measured** feed concentration, not the nominal one: the capacity calculation in Part 2 depends on it directly.

`VOLUME_AT` says where in each fraction the volume in your file was recorded. If it is the cumulative volume at the **end** of each fraction, the notebook plots each concentration at the middle of its fraction, because the concentration you measured is an average over the whole fraction.

In [ ]:
DATA_FILE = "breakthrough_data.csv"

# Column headers in the data file.
VOLUME = "Volume (mL)"
CONC = "Concentration (mg/mL)"
FLOW = "Flow Rate (mL/min)"
FEED = "Inlet Concentration (mg/mL)"
RUN = "Run ID"

# "end" if the volume is the cumulative volume at the end of each fraction, "middle" if at its middle.
VOLUME_AT = "end"

# Column: Cytiva HiTrap 5 mL (packed bed 1.6 cm diameter x 2.5 cm high).
BED_VOLUME = 5.0    # mL of packed bed
BED_HEIGHT = 2.5    # cm
POROSITY = 0.4      # fraction of the bed volume that is liquid between the beads

# Langmuir constant, which sets how sharp the breakthrough front is in the simulation.
K_L = 0.313         # mL/mg

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import curve_fit
from scipy.sparse import diags, bmat
from scipy.stats import t as t_dist

warnings.filterwarnings("ignore", message=".*parameter as float was deprecated.*")

if not os.path.exists(DATA_FILE):
    csv_files = sorted(f for f in os.listdir(".") if f.lower().endswith(".csv"))
    raise FileNotFoundError(
        f'Cannot find "{DATA_FILE}" in the folder this notebook is in.\n'
        f"CSV files in this folder: {', '.join(csv_files) if csv_files else 'none'}.\n"
        "Move your data file into this folder, or change DATA_FILE in the settings cell to match its name exactly.")
df = pd.read_csv(DATA_FILE)
missing_cols = [c for c in (VOLUME, CONC, FLOW, FEED, RUN) if c not in df.columns]
if missing_cols:
    raise KeyError(f"Column(s) {missing_cols} not found. The file's columns are: {list(df.columns)}.")
if VOLUME_AT not in ("end", "middle"):
    raise ValueError('VOLUME_AT must be "end" or "middle".')
df[RUN] = df[RUN].astype(str).str.strip()

AREA = BED_VOLUME / BED_HEIGHT      # cm², cross-section of the bed
NZ = 60                             # slices along the column; they also add a small amount of axial spreading

runs = []
for run_id, g in df.groupby(RUN, sort=False):
    g = g.sort_values(VOLUME)
    V = g[VOLUME].to_numpy(float)
    c = g[CONC].to_numpy(float)
    dV = np.diff(np.r_[0.0, V])                       # volume of each fraction
    V_plot = V - dV / 2 if VOLUME_AT == "end" else V  # middle of each fraction
    runs.append(dict(id=run_id, Q=float(g[FLOW].iloc[0]), c0=float(g[FEED].iloc[0]),
                     V=V, V_plot=V_plot, dV=dV, c=c))


def simulate(Q, c0, q_max, Ka, V_out):
    # Outlet concentration of the column at the volumes V_out (mL), for a step change to feed c0 at V = 0.
    dz = BED_HEIGHT / NZ
    u = Q / (POROSITY * AREA)          # interstitial velocity, cm/min

    def rhs(t, y):
        c, q = y[:NZ], y[NZ:]
        dq = Ka * (q_max * K_L * c / (1 + K_L * c) - q)
        c_up = np.r_[c0, c[:-1]]
        dc = -u * (c - c_up) / dz - dq / POROSITY
        return np.r_[dc, dq]

    # Which variables each equation depends on, so the solver can work efficiently.
    band = diags([1.0, 1.0], [0, -1], shape=(NZ, NZ))
    eye = diags([1.0], [0], shape=(NZ, NZ))
    sparsity = bmat([[band, eye], [eye, eye]])
    t_out = np.asarray(V_out, float) / Q
    sol = solve_ivp(rhs, [0, t_out.max()], np.zeros(2 * NZ), t_eval=t_out, method="BDF",
                    jac_sparsity=sparsity, rtol=1e-6, atol=1e-8)
    return sol.y[NZ - 1]


for r in runs:
    print(f"{r['id']}: {len(r['c'])} fractions, flow {r['Q']:g} mL/min, feed {r['c0']:g} mg/mL, "
          f"{r['V'].max():g} mL loaded.")

## Part 1. Breakthrough curves

The left plot shows the outlet concentration against the volume loaded. The right plot shows the same data as a fraction of the feed concentration, $c/c_0$, so runs with different feeds can be compared. A curve that levels off below $c/c_0 = 1$ means either the column is not yet saturated or the feed concentration in your data file is higher than the real one.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for r in runs:
    axes[0].plot(r["V_plot"], r["c"], "o-", ms=4, label=r["id"])
    axes[1].plot(r["V_plot"], r["c"] / r["c0"], "o-", ms=4, label=r["id"])
axes[0].set_ylabel("Outlet concentration (mg/mL)")
axes[1].set_ylabel("c / c0")
axes[1].axhline(1, color="gray", linestyle="--", linewidth=1)
for ax in axes:
    ax.set_xlabel("Volume loaded (mL)")
    ax.legend(frameon=False)
fig.tight_layout()
plt.show()

## Part 2. Column capacity from a mass balance

Everything you pumped in either came out in the fractions, stayed in the liquid inside the column, or bound to the resin:

$$\text{protein bound} = \underbrace{c_0\,V_{\text{loaded}}}_{\text{pumped in}} \;-\; \underbrace{\sum c_i\,\Delta V_i}_{\text{collected in fractions}} \;-\; \underbrace{c_0\,\varepsilon\,V_{\text{bed}}}_{\text{liquid in the column}}$$

where $c_i$ is the concentration in fraction $i$ and $\Delta V_i$ its volume. The last term assumes the liquid in the column is at the feed concentration at the end of the run. The **capacity** is the protein bound per mL of packed bed:

$$q = \frac{\text{protein bound}}{V_{\text{bed}}} \quad \text{(mg/mL resin)}$$

**Self-check.** The plot shows the protein bound as loading proceeds. Once the column is saturated, it should level off. If it is still rising steadily at the end of the run while the outlet concentration has leveled off, the feed concentration in your data file is probably too high: every mL of feed then adds protein that never really existed. If the outlet never approached the feed, the column was not saturated and your capacity is a lower bound.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.5))
rows = []
for r in runs:
    collected = np.cumsum(r["c"] * r["dV"])
    bound = r["c0"] * r["V"] - collected - r["c0"] * POROSITY * BED_VOLUME
    r["q"] = bound[-1] / BED_VOLUME
    final = r["c"][-3:].mean() / r["c0"]
    rows.append({"Run": r["id"], "Flow (mL/min)": r["Q"], "Feed c0 (mg/mL)": r["c0"],
                 "Loaded (mL)": r["V"].max(), "Final c/c0": round(final, 3),
                 "Protein bound (mg)": round(bound[-1], 1), "Capacity q (mg/mL resin)": round(r["q"], 1)})
    ax.plot(r["V"], bound, "o-", ms=4, label=r["id"])
ax.set_xlabel("Volume loaded (mL)")
ax.set_ylabel("Protein bound (mg)")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

capacity = pd.DataFrame(rows)
print(capacity.to_string(index=False))
for r, row in zip(runs, rows):
    if row["Final c/c0"] < 0.9:
        print(f"\n{r['id']}: the outlet reached only {row['Final c/c0']:.0%} of the feed. Either the column was not "
              "saturated (the capacity is then a lower bound), or the feed concentration in your file is too high.")

## Part 3. The mass-transfer coefficient $K_a$

**The model.** The model is a differential mass balance on a thin slice of the column, of thickness $dz$ and cross-section $A$. Protein accumulates in the liquid between the beads and on the resin, and the difference between what flows in and what flows out supplies it:

$$\underbrace{\varepsilon A\,dz\,\frac{\partial c}{\partial t}}_{\text{in the liquid}} + \underbrace{A\,dz\,\frac{\partial q}{\partial t}}_{\text{on the resin}} = \underbrace{Q\,c\big|_{z} - Q\,c\big|_{z+dz}}_{\text{in} \;-\; \text{out}}$$

Dividing by $\varepsilon A\,dz$, with the interstitial velocity $u = Q/(\varepsilon A)$, gives

$$\frac{\partial c}{\partial t} = -u\,\frac{\partial c}{\partial z} \;-\; \frac{1}{\varepsilon}\,\frac{\partial q}{\partial t}, \qquad \frac{\partial q}{\partial t} = K_a\,\big(q^*(c) - q\big), \qquad q^*(c) = \frac{q_{max}\,K_L\,c}{1 + K_L\,c}$$

Here $c$ is the protein concentration in the liquid, $q$ the protein bound per mL of bed, and $q^*(c)$ the amount the resin would hold at equilibrium with the liquid around it. The rate of binding is proportional to how far the resin is from equilibrium. $K_a$ (1/min) is the **mass-transfer coefficient**: it lumps together transfer through the liquid film around each bead and diffusion into the pores. A large $K_a$ gives a sharp breakthrough; a small $K_a$ gives a gradual one. $q_{max}$ is set so that $q^*(c_0)$ equals the capacity from Part 2, so the simulated breakthrough happens at the right volume.

The notebook solves these equations by dividing the column into 60 slices.

**The fit.** $K_a$ appears inside the simulation, so it cannot be found by linear regression. It is found by nonlinear least squares: the notebook adjusts $K_a$, simulates the column each time, and finds the value that makes the sum of squared residuals (SSR) between the simulated and measured outlet concentrations as small as possible. The plots show each run with its best-fit simulation.

In [ ]:
def make_model(r):
    # Simulated outlet concentration for run r at volumes V, as a function of Ka.
    return lambda V, Ka: simulate(r["Q"], r["c0"], r["q_max"], Ka, V)

for r in runs:
    r["q_max"] = r["q"] * (1 + K_L * r["c0"]) / (K_L * r["c0"])
    model = make_model(r)
    r["model"] = model

    # Coarse search on a log scale for a starting value, then nonlinear least squares.
    grid = np.logspace(-2, 1.3, 12)
    Ka0 = grid[int(np.argmin([np.sum((model(r["V_plot"], k) - r["c"])**2) for k in grid]))]
    popt, pcov = curve_fit(model, r["V_plot"], r["c"], p0=[Ka0], bounds=(1e-4, 100), diff_step=1e-3)

    r["Ka"] = popt[0]
    r["fit"] = model(r["V_plot"], r["Ka"])
    r["n"] = len(r["c"])
    r["ssr"] = np.sum((r["c"] - r["fit"])**2)
    r["s2"] = r["ssr"] / (r["n"] - 1)              # average squared residual (MSE), n - 1 degrees of freedom
    r["se"] = np.sqrt(pcov[0, 0])                  # standard error of Ka
    r["t"] = t_dist.ppf(0.975, r["n"] - 1)
    r["ci"] = r["t"] * r["se"]                     # half-width of the 95% CI

fig, axes = plt.subplots(1, len(runs), figsize=(6 * len(runs), 4.3), squeeze=False)
for r, ax in zip(runs, axes[0]):
    V_fine = np.linspace(r["V_plot"].min() * 0.5, r["V"].max(), 200)
    ax.plot(r["V_plot"], r["c"], "o", color="black", ms=4, label="Measured")
    ax.plot(V_fine, r["model"](V_fine, r["Ka"]), "-", color="tab:blue", label=f"Simulation, Ka = {r['Ka']:.3g} 1/min")
    ax.axhline(r["c0"], color="gray", linestyle="--", linewidth=1)
    ax.set_xlabel("Volume loaded (mL)")
    ax.set_ylabel("Outlet concentration (mg/mL)")
    ax.set_title(r["id"], fontsize=11)
    ax.legend(frameon=False)
fig.tight_layout()
plt.show()

## Part 4. Confidence interval for $K_a$

Your best-fit $K_a$ comes from one set of noisy measurements, so it is reported with a 95% confidence interval: estimate $\pm\ t \times se$, where $se$ is the standard error and $t$ comes from the t-distribution.

**Where the standard error comes from.** You have seen the standard error of a straight line's slope:

$$se(\hat\beta_1) = \sqrt{\frac{s^2}{\sum (x_i - \bar x)^2}}, \qquad s^2 = \frac{\sum (y_i - \hat y_i)^2}{n - 2}$$

The numerator, $s^2$, is the average squared residual: how much the data scatter around the fit. The denominator is how strongly the fit responds to the slope: if the slope changes by $\Delta\beta_1$, each fitted value moves by $(x_i - \bar x)\,\Delta\beta_1$. So the standard error is the scatter of the data divided by how strongly the fitted values respond to the parameter.

The column simulation uses the same idea for $K_a$:

$$se(\hat K_a) = \sqrt{\frac{s^2}{\sum \left(\Delta \hat c_i / \Delta K_a\right)^2}}, \qquad s^2 = \frac{\sum (c_i - \hat c_i)^2}{n - 1}$$

Here $c_i$ is the measured outlet concentration in fraction $i$, $\hat c_i$ the simulated one, and $n$ the number of fractions. The denominator of $s^2$ is $n - 1$ because one parameter, $K_a$, is fitted. There is no formula for $\Delta \hat c_i / \Delta K_a$, so the notebook changes $K_a$ slightly, reruns the simulation, and measures how far each simulated concentration moves. Fractions on the rising part of the breakthrough curve usually move the most, and fractions on the plateau barely move, so the fractions that capture the rise do most of the work of pinning $K_a$ down.

In [ ]:
for r in runs:
    print(f"{r['id']}: {r['n']} fractions, t = {r['t']:.3f}")
    print(f"   Ka = {r['Ka']:.3g} ± {r['ci']:.2g} 1/min   (95% CI {r['Ka'] - r['ci']:.3g} to {r['Ka'] + r['ci']:.3g})\n")

## Part 5. Sensitivity of the fit to $K_a$

Each plot below changes $K_a$ around its best-fit value and shows how much worse the fit gets, measured by the sum of squared residuals (SSR). The SSR is lowest at the best-fit value and rises on either side.

The confidence interval in Part 4 is an approximation. It uses how sharply the SSR curves upward at the best-fit value and assumes the curve keeps that same curvature everywhere, like a parabola. These plots check that approximation.

- The **solid curve** is the SSR calculated by rerunning the simulation with your data at each value. This is the real SSR curve.
- The **dotted parabola** is the SSR curve the confidence interval assumes.
- The **shaded band** is the 95% confidence interval from Part 4.
- The **dashed line** marks the height of the parabola at the edges of the interval: how much worse the fit is allowed to get before you leave the interval.

**Interpreting your results.** Look first at how sharply each SSR curve rises on either side of its minimum. This curvature is the denominator of the standard error in Part 4: how strongly the simulated concentrations respond when $K_a$ changes. If they respond strongly, even a small change in $K_a$ makes the fit noticeably worse, so the curve rises sharply and the interval is narrow. If they barely respond, the curve is broad and flat near its minimum, and a wider range of values fits nearly as well. Noisier data (a larger MSE, the numerator) raise the dashed line and widen the interval.

Then check the approximation. Follow the dashed line and see where the solid curve crosses it. If it crosses at the edges of the shaded band, the real SSR curve matches the parabola and the interval is reliable. If it crosses outside the band on one side, the real curve rises more slowly there, and the interval should extend further on that side.

Finally, the interval assumes the residuals are random scatter, which Part 6 checks. A narrow interval means your data pin $K_a$ down for this model, not that the model is right. If the residuals show a pattern, the true uncertainty is larger than the interval.

**Questions to consider**

1. Which run pins $K_a$ down more tightly? Compare each interval's half-width as a percentage of its best-fit value.
2. For each run, does the solid curve follow the dotted parabola on both sides? If not, on which side should the interval extend further?

In [ ]:
fig, axes = plt.subplots(1, len(runs), figsize=(6 * len(runs), 4.4), squeeze=False)
for r, ax in zip(runs, axes[0]):
    Ka, half = r["Ka"], r["ci"]
    threshold = r["ssr"] + r["t"]**2 * r["s2"]    # height of the parabola at the edges of the 95% CI
    k_line = np.linspace(max(Ka - 2 * half, 0.2 * Ka), Ka + 2 * half, 31)
    ssr_line = [np.sum((r["model"](r["V_plot"], k) - r["c"])**2) for k in k_line]
    parabola = r["ssr"] + r["s2"] * ((k_line - Ka) / r["se"])**2

    ax.axvspan(Ka - half, Ka + half, color="tab:blue", alpha=0.15, label="95% CI")
    ax.plot(k_line, parabola, ":", color="gray", linewidth=2, label="Parabola assumed by the CI")
    ax.plot(k_line, ssr_line, "-", color="black", label="SSR from rerunning the simulation")
    ax.axhline(threshold, color="tab:red", linestyle="--", label="SSR at edges of 95% CI")
    ax.axvline(Ka, color="gray", linewidth=0.8)
    ax.set_title(f"{r['id']}: Ka = {Ka:.3g} ± {half:.2g} 1/min", fontsize=11)
    ax.set_xlabel("Ka (1/min)")
    ax.set_ylabel("SSR")

handles, labels = axes[0][0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=4, frameon=False, fontsize=9)
fig.subplots_adjust(left=0.08, right=0.98, bottom=0.25, top=0.92, wspace=0.3)
plt.show()

## Part 6. Residuals

A residual is the measured outlet concentration minus the simulated one. If the model describes your column, the residuals scatter randomly above and below zero along the whole breakthrough curve. A run of residuals on one side of zero, or a steady trend, means the model is missing something about how protein moved through the column.

**Questions to consider**

1. Are the residuals scattered randomly about zero, or do they form stretches above or below zero? Where along the breakthrough curve does the simulation miss most?
2. If the residuals show a pattern, what about how protein moves into the resin might a single mass-transfer coefficient leave out?
3. Does any single fraction stand out from the rest?
4. Do the residuals near the end of the run sit above or below zero? What would that say about the feed concentration in your data file?

In [ ]:
fig, axes = plt.subplots(1, len(runs), figsize=(6 * len(runs), 3.8), squeeze=False)
for r, ax in zip(runs, axes[0]):
    ax.axhline(0, color="gray", linestyle="--", linewidth=1)
    ax.plot(r["V_plot"], r["c"] - r["fit"], "o-", color="black", ms=4)
    ax.set_xlabel("Volume loaded (mL)")
    ax.set_ylabel("Measured − simulated (mg/mL)")
    ax.set_title(f"{r['id']}: residuals", fontsize=11)
fig.tight_layout()
plt.show()

## Part 7. Summary

The table collects the results for each run and is saved as `breakthrough_results.csv` for formatting in your report. When you compare runs, note what differs between them: if two runs differ in both flow rate and feed concentration, a difference in $K_a$ or capacity cannot be assigned to either one alone.

In [ ]:
summary = pd.DataFrame([{
    "Run": r["id"], "Flow (mL/min)": r["Q"], "Feed c0 (mg/mL)": r["c0"],
    "Capacity q (mg/mL resin)": round(r["q"], 1),
    "Ka (1/min)": round(r["Ka"], 3), "Ka 95% CI (±)": round(r["ci"], 3)} for r in runs])
print(summary.to_string(index=False))
summary.to_csv("breakthrough_results.csv", index=False)